In [1]:
import optuna
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import StandardScaler

from adbench.run_new import RunPipeline
from adbench.myutils_new import Utils
from MSML_v9 import MSML


# -------------------------------------------------
# Optuna Objective (FULL ADBench protocol)
# -------------------------------------------------
def objective(trial):

    # ---------- Hyperparameter Search Space ----------
    k = trial.suggest_int("k", 10, 100)
    nbd_sample_count_threshold = trial.suggest_int(
        "nbd_sample_count_threshold", 5, 80
    )
    learning_rate = trial.suggest_float(
        "learning_rate", 0.05, 1.0, log=True
    )
    max_iters_shift = trial.suggest_int("max_iters_shift", 5, 20)
    shift_threshold = trial.suggest_float(
        "shift_threshold", 1e-5, 1e-2, log=True
    )
    anomalyThreshold = trial.suggest_float(
        "anomalyThreshold", 0.01, 0.3
    )

    # ---------- Customized MSML Wrapper ----------
    class OptunaMSML(MSML):
        def __init__(self, seed, model_name=None):
            super().__init__(
                seed=seed,
                k=k,
                nbd_sample_count_threshold=nbd_sample_count_threshold,
                learning_rate=learning_rate,
                max_iters_shift=max_iters_shift,
                shift_threshold=shift_threshold,
                anomalyThreshold=anomalyThreshold,
                scaler=StandardScaler()
            )

    # ---------- ADBench Pipeline (FULL SETTING) ----------
    pipeline = RunPipeline(
        suffix="Optuna_MSML_FULL",
        parallel="unsupervise",
        realistic_synthetic_mode="local",
        noise_type=None
    )

    # ---------- Run FULL ADBench ----------
    pipeline.run(clf=OptunaMSML)

    # ---------- Load AUCROC Results ----------
    result_path = os.path.join(
    "adbench",
    "result",
    f"AUCROC_{pipeline.suffix}.csv"
    )

    df_aucroc = pd.read_csv(result_path, index_col=0)

    # ---------- Compute Mean AUCROC ----------
    mean_aucroc = np.nanmean(df_aucroc.values)

    return float(mean_aucroc)


# -------------------------------------------------
# Optuna Callback (print after each trial)
# -------------------------------------------------
def print_trial_result(study, trial):
    print("\n================ Trial Finished ================")
    print(f"Trial number : {trial.number}")
    print(f"AUCROC       : {trial.value}")
    print("Hyperparameters:")
    for k, v in trial.params.items():
        print(f"  {k}: {v}")
    print("================================================\n")


# -------------------------------------------------
# Main
# -------------------------------------------------
if __name__ == "__main__":

    utils = Utils()
    utils.download_datasets()

    study = optuna.create_study(
        direction="maximize",
        study_name="MSML_AUCROC_ADBench_FULL"
    )

    study.optimize(
        objective,
        n_trials=10,
        callbacks=[print_trial_result]
    )

    print(" Best AUCROC:", study.best_value)
    print(" Best hyperparameters:", study.best_params)

    df = study.trials_dataframe()
    df.to_csv(
        "adbench/result/MSDE_optuna_local_none_noise.csv",
        index=False
    )

    print(" Saved to adbench/result/MSDE_optuna_local_none_noise.csv")


if there is any question while downloading datasets, we suggest you to download it from the website:
https://github.com/Minqi824/ADBench/tree/main/adbench/datasets
如果您在中国大陆地区，请使用链接：
https://jihulab.com/BraudoCC/ADBench_datasets/
100% [................................................................................] 3852 / 3852

100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 353.17it/s]
[I 2026-01-08 16:46:03,415] A new study created in memory with name: MSML_AUCROC_ADBench_FULL


CIFAR10_0.npz already exists. Skipping download...
CIFAR10_1.npz already exists. Skipping download...
CIFAR10_2.npz already exists. Skipping download...
CIFAR10_3.npz already exists. Skipping download...
CIFAR10_4.npz already exists. Skipping download...
CIFAR10_5.npz already exists. Skipping download...
CIFAR10_6.npz already exists. Skipping download...
CIFAR10_7.npz already exists. Skipping download...
CIFAR10_8.npz already exists. Skipping download...
CIFAR10_9.npz already exists. Skipping download...
FashionMNIST_0.npz already exists. Skipping download...
FashionMNIST_1.npz already exists. Skipping download...
FashionMNIST_2.npz already exists. Skipping download...
FashionMNIST_3.npz already exists. Skipping download...
FashionMNIST_4.npz already exists. Skipping download...
FashionMNIST_5.npz already exists. Skipping download...
FashionMNIST_6.npz already exists. Skipping download...
FashionMNIST_7.npz already exists. Skipping download...
FashionMNIST_8.npz already exists. Skippin

0it [00:00, ?it/s]

generating duplicate samples for dataset 15_Hepatitis...


...

Current experiment parameters: ('9_census', 0.0, np.int64(1)), model: Customized, metrics: {'aucroc': np.float64(1.0), 'aucpr': np.float64(1.0), 'p_at_n': np.float64(1.0), 'adj_p_at_n': np.float64(1.0), 'adj_ap': np.float64(1.0)}, fitting time: 1.1920928955078125e-06, inference time: 43.99729561805725
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(628), 'Anomalies Ratio(%)': np.float64(6.28)}
Model: Customized, AUC-ROC: 0.999990542053812, AUC-PR: 0.9998621982140888


1082it [24:13,  7.02s/it]

Current experiment parameters: ('9_census', 0.0, np.int64(2)), model: Customized, metrics: {'aucroc': np.float64(0.999990542053812), 'aucpr': np.float64(0.9998621982140888), 'p_at_n': np.float64(0.9946808510638298), 'adj_p_at_n': np.float64(0.9943252322871583), 'adj_ap': np.float64(0.9998529852924134)}, fitting time: 1.1920928955078125e-06, inference time: 44.20843696594238
subsampling for dataset 9_census...
current noise type: None
{'Samples': 10000, 'Features': 500, 'Anomalies': np.int64(652), 'Anomalies Ratio(%)': np.float64(6.52)}
Model: Customized, AUC-ROC: 0.9943102419284405, AUC-PR: 0.9486168982195378


1104it [26:07,  1.42s/it]
[I 2026-01-08 20:42:56,425] Trial 9 finished with value: 0.9043220512306429 and parameters: {'k': 92, 'nbd_sample_count_threshold': 79, 'learning_rate': 0.05278850379719878, 'max_iters_shift': 18, 'shift_threshold': 1.792769163533591e-05, 'anomalyThreshold': 0.11831309394540832}. Best is trial 8 with value: 0.9117932782919781.


Current experiment parameters: ('9_census', 0.0, np.int64(3)), model: Customized, metrics: {'aucroc': np.float64(0.9943102419284405), 'aucpr': np.float64(0.9486168982195378), 'p_at_n': np.float64(0.8520408163265306), 'adj_p_at_n': np.float64(0.8416984482808816), 'adj_ap': np.float64(0.9450252120751117)}, fitting time: 1.6689300537109375e-06, inference time: 47.98979115486145

================ Trial Finished ================
Trial number : 9
AUCROC       : 0.9043220512306429
Hyperparameters:
  k: 92
  nbd_sample_count_threshold: 79
  learning_rate: 0.05278850379719878
  max_iters_shift: 18
  shift_threshold: 1.792769163533591e-05
  anomalyThreshold: 0.11831309394540832

 Best AUCROC: 0.9117932782919781
 Best hyperparameters: {'k': 26, 'nbd_sample_count_threshold': 32, 'learning_rate': 0.13028892905053382, 'max_iters_shift': 13, 'shift_threshold': 2.9828058695323927e-05, 'anomalyThreshold': 0.044987609728108266}
 Saved to adbench/result/MSDE_optuna_local_none_noise.csv
